In [1]:
!pip install -q litellm requests pydantic-ai gradio python-dotenv

import os
os.environ["GROQ_API_KEY"]   = "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"
os.environ["SERPER_API_KEY"] = "b8690701a0014c84f94e757e24dd7633d55d5489"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.4/93.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.2/101.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.4/705.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 627.7/627.7 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from pydantic import BaseModel
from pydantic_ai import Agent
class TriageSummary(BaseModel):
    primary_symptoms: list[str]
    potential_causes: list[str]
    urgency_level: str
    recommended_specialist: str
    general_advice: str
    disclaimer: str
triage_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    output_type=TriageSummary,
    system_prompt=(
        "You are a professional healthcare triage assistant. "
        "Analyze the user's symptoms and return a structured JSON assessment. "
        "Always include a strict medical disclaimer stating you are an AI, not a doctor."
    ),
)

In [11]:
import requests
import os
import gradio as gr
from pydantic_ai import Agent
HEALTH_SYSTEM_PROMPT = """
You are a highly capable AI Healthcare & Wellness Assistant.
You can help users understand symptoms, find general medical information, check drug interactions, and find nearby hospitals.

Rules:
1. If the user asks about symptoms or conditions, use 'search_medical_info'.
2. If the user provides weight and height, use 'calculate_bmi'.
3. ALWAYS end your responses with this disclaimer: "*Disclaimer: I am an AI, not a medical professional. Please consult a doctor for proper diagnosis and treatment.*"
4. EMERGENCIES: If a condition sounds life-threatening (e.g., chest pain, severe bleeding, difficulty breathing), immediately advise them to call emergency services. If they provide their location, use 'find_nearby_hospitals'.
5. Provide direct, comprehensive answers based on the user's input. DO NOT ask follow-up questions or ask for more details. Give the best possible assessment or advice with the information provided.
6. If the user asks about taking multiple medications together, use 'check_drug_interaction'.
"""
healthcare_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=HEALTH_SYSTEM_PROMPT,
)

# Tool 1: General Medical Search
@healthcare_agent.tool_plain
def search_medical_info(query: str) -> str:
    """Search the web for up-to-date medical articles and health guidelines."""
    url = "https://google.serper.dev/search"
    headers = {
        "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
        "Content-Type": "application/json",
    }
    response = requests.post(url, headers=headers, json={"q": query + " medical authoritative", "num": 3})

    if response.status_code != 200:
        return "Search tool unavailable."

    results = []
    for item in response.json().get("organic", [])[:3]:
        title   = item.get("title", "")
        snippet = item.get("snippet", "")
        results.append(f"- {title}\n  {snippet}")

    return "\n".join(results) if results else "No clear medical information found."

# Tool 2: BMI Calculator
@healthcare_agent.tool_plain
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculate Body Mass Index (BMI)."""
    if height_m <= 0:
        return "Invalid height."
    bmi = weight_kg / (height_m ** 2)
    return f"Calculated BMI is {bmi:.2f}."

# Tool 3: Drug Interaction Checker
@healthcare_agent.tool_plain
def check_drug_interaction(drug1: str, drug2: str) -> str:
    """Check if there are known medical interactions between two medications."""
    url = "https://google.serper.dev/search"
    headers = {
        "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
        "Content-Type": "application/json",
    }
    query = f"{drug1} and {drug2} drug interaction side effects medical"
    response = requests.post(url, headers=headers, json={"q": query, "num": 3})

    if response.status_code != 200:
        return "Interaction check unavailable."

    results = []
    for item in response.json().get("organic", [])[:3]:
        snippet = item.get("snippet", "")
        results.append(f"- {snippet}")

    return "Potential Interaction Data:\n" + "\n".join(results) if results else "No specific interaction found, but consult a doctor."

# Tool 4: Hospital Locator
@healthcare_agent.tool_plain
def find_nearby_hospitals(location: str) -> str:
    """Find nearby hospitals or emergency rooms based on the user's location."""
    url = "https://google.serper.dev/search"
    headers = {
        "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
        "Content-Type": "application/json",
    }
    query = f"top emergency hospitals near {location}"
    response

In [12]:
import gradio as gr

# Dedicated streaming agent for Gradio UI
streaming_health_agent = Agent(
    model="groq:llama-3.3-70b-versatile",
    system_prompt=HEALTH_SYSTEM_PROMPT,
)

async def stream_health_response(user_message: str, _history):
    """Async generator — streams the response live."""
    partial = ""
    async with streaming_health_agent.run_stream(user_message) as result:
        async for text in result.stream_text(delta=False):
            partial = text
            yield partial

def build_health_ui():
    with gr.Blocks(title="AI Health Assistant", theme=gr.themes.Soft(primary_hue="blue")) as demo:
        gr.Markdown("## 🩺 AI Healthcare & Wellness Assistant (Level 3)")
        gr.Markdown(
            "Ask me about symptoms, drug interactions, or find nearby hospitals. "
            "I will ask follow-up questions if I need more details!"
        )
        gr.ChatInterface(
            fn=stream_health_response,
            examples=[
                "I have a stomach ache.",
                "Is it safe to take Ibuprofen and Lisinopril together?",
                "I am having severe chest pain. I live in Andheri, Mumbai. What should I do?",
                "My weight is 75kg and height is 1.75m. Is that healthy?"
            ],
        )
    return demo

build_health_ui().launch(share=True)

/tmp/ipykernel_8911/1778566238.py:18: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Health Assistant", theme=gr.themes.Soft(primary_hue="blue")) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eb04d33a90d02ad830.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
